In [0]:
# Questo notebook inizializza la parte meteo del progetto.
#
# Legge le città dalla tabella:
# progetto_meteo.metadati.citta_monitorate
#
# Poi scarica i dati meteorologici da Open-Meteo:
# - storico dal 01/01/2024 fino a ieri;
# - giornata corrente fino all'ultima ora completa.
#
# Il recupero della giornata corrente serve a evitare un buco tra mezzanotte
# e l'orario in cui viene avviato il progetto.
#
# Il risultato viene salvato nella tabella Bronze:
# progetto_meteo.bronze.meteo_storico
#
# In questa tabella la temperatura oraria viene salvata come Temp_Oraria.

import requests

from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    LongType,
    DoubleType,
    TimestampType
)


# Imposto catalogo, schemi e tabelle del progetto.
catalogo = "progetto_meteo"
schema_metadati = "metadati"
schema_bronze = "bronze"

tabella_citta = f"{catalogo}.{schema_metadati}.citta_monitorate"
tabella_storico = f"{catalogo}.{schema_bronze}.meteo_storico"


# Definisco lo schema della tabella meteo_storico.
# Uso uno schema esplicito per evitare che Spark interpreti i tipi in modo automatico.
schema_storico = StructType([
    StructField("Citta", StringType(), True),
    StructField("Regione", StringType(), True),
    StructField("Area", StringType(), True),
    StructField("Data", DateType(), True),
    StructField("Ora", LongType(), True),
    StructField("Temp_Max", DoubleType(), True),
    StructField("Temp_Min", DoubleType(), True),
    StructField("Temp_Oraria", DoubleType(), True),
    StructField("Umidita", DoubleType(), True),
    StructField("Velocita_Vento", DoubleType(), True),
    StructField("Precipitazioni", DoubleType(), True),
    StructField("Timestamp", TimestampType(), True)
])


# Converto i numeri ricevuti dall'API in float.
# Se il valore manca, lascio None.
def numero(valore):
    if valore is None:
        return None
    return float(valore)


# Trasformo i dati ricevuti da Open-Meteo in righe compatibili con la tabella Bronze.
# La funzione viene usata sia per lo storico sia per il recupero della giornata corrente.
def aggiungi_righe_storico(righe_storico, lista_citta, lista_risposte_api, timestamp, data_limite=None, ora_massima=None):

    for citta, dati_citta in zip(lista_citta, lista_risposte_api):

        orari = dati_citta["hourly"]
        giornalieri = dati_citta["daily"]

        mappa_giorni = {}

        for giorno, temp_max, temp_min in zip(
            giornalieri["time"],
            giornalieri["temperature_2m_max"],
            giornalieri["temperature_2m_min"]
        ):
            mappa_giorni[giorno] = {
                "Temp_Max": temp_max,
                "Temp_Min": temp_min
            }

        for valore_ora, temperatura, umidita, precipitazione, vento in zip(
            orari["time"],
            orari["temperature_2m"],
            orari["relative_humidity_2m"],
            orari["precipitation"],
            orari["wind_speed_10m"]
        ):

            data_ora = datetime.fromisoformat(valore_ora)
            giorno = str(data_ora.date())

            if data_limite is not None and data_ora.date() != data_limite:
                continue

            if ora_massima is not None and data_ora.hour > ora_massima:
                continue

            dati_giorno = mappa_giorni.get(giorno, {
                "Temp_Max": None,
                "Temp_Min": None
            })

            righe_storico.append({
                "Citta": citta["citta"],
                "Regione": citta["regione"],
                "Area": citta["area"],
                "Data": data_ora.date(),
                "Ora": data_ora.hour,
                "Temp_Max": numero(dati_giorno["Temp_Max"]),
                "Temp_Min": numero(dati_giorno["Temp_Min"]),
                "Temp_Oraria": numero(temperatura),
                "Umidita": numero(umidita),
                "Velocita_Vento": numero(vento),
                "Precipitazioni": numero(precipitazione),
                "Timestamp": timestamp
            })


# Seleziono il catalogo del progetto.
spark.sql(f"USE CATALOG {catalogo}")


# Leggo le città da monitorare.
# Queste città contengono coordinate e macroarea finale del progetto.
df_citta = (
    spark.table(tabella_citta)
    .select(
        "citta",
        "regione",
        "area",
        "latitudine",
        "longitudine"
    )
    .orderBy("citta")
)

lista_citta = [riga.asDict() for riga in df_citta.collect()]


# Calcolo il periodo storico principale.
# Lo storico parte dal 01/01/2024 e arriva fino a ieri.
adesso = datetime.now(ZoneInfo("Europe/Rome")).replace(microsecond=0)
oggi = adesso.date()

data_fine_storico = oggi - timedelta(days=1)
data_inizio_storico = datetime(2024, 1, 1).date()

ultima_ora_completa = adesso.hour - 1
timestamp = adesso.replace(tzinfo=None)

print(f"Periodo storico principale: da {data_inizio_storico} a {data_fine_storico}")

if ultima_ora_completa >= 0:
    print(f"Recupero giornata di avvio: {oggi}, ore 00-{ultima_ora_completa}")
else:
    print("Nessuna ora della giornata corrente da recuperare.")


# Preparo latitudini e longitudini per una sola chiamata multi-città.
# Open-Meteo permette di passare più coordinate nella stessa richiesta.
latitudini = ",".join([str(citta["latitudine"]) for citta in lista_citta])
longitudini = ",".join([str(citta["longitudine"]) for citta in lista_citta])


# Preparo la lista finale.
# Qui finiranno sia le righe storiche sia le righe della giornata corrente recuperata.
righe_storico = []


# Chiamata alla Historical API di Open-Meteo.
# Scarico i dati orari e giornalieri dal 01/01/2024 fino a ieri.
endpoint_storico = "https://archive-api.open-meteo.com/v1/archive"

parametri_storico = {
    "latitude": latitudini,
    "longitude": longitudini,
    "start_date": str(data_inizio_storico),
    "end_date": str(data_fine_storico),
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "daily": "temperature_2m_max,temperature_2m_min",
    "timezone": "Europe/Rome"
}

risposta_storico = requests.get(endpoint_storico, params=parametri_storico, timeout=120)
risposta_storico.raise_for_status()

dati_storico = risposta_storico.json()
lista_storico_api = dati_storico if isinstance(dati_storico, list) else [dati_storico]

aggiungi_righe_storico(
    righe_storico=righe_storico,
    lista_citta=lista_citta,
    lista_risposte_api=lista_storico_api,
    timestamp=timestamp
)


# Chiamata alla Forecast API di Open-Meteo.
# Serve solo per recuperare le ore già complete della giornata corrente.
if ultima_ora_completa >= 0:

    endpoint_forecast = "https://api.open-meteo.com/v1/forecast"

    parametri_forecast = {
        "latitude": latitudini,
        "longitude": longitudini,
        "start_date": str(oggi),
        "end_date": str(oggi),
        "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
        "daily": "temperature_2m_max,temperature_2m_min",
        "timezone": "Europe/Rome"
    }

    risposta_forecast = requests.get(endpoint_forecast, params=parametri_forecast, timeout=120)
    risposta_forecast.raise_for_status()

    dati_forecast = risposta_forecast.json()
    lista_forecast_api = dati_forecast if isinstance(dati_forecast, list) else [dati_forecast]

    aggiungi_righe_storico(
        righe_storico=righe_storico,
        lista_citta=lista_citta,
        lista_risposte_api=lista_forecast_api,
        timestamp=timestamp,
        data_limite=oggi,
        ora_massima=ultima_ora_completa
    )


# Creo il DataFrame finale usando lo schema esplicito.
df_storico = spark.createDataFrame(righe_storico, schema=schema_storico)


# Salvo tutto nella tabella Delta Bronze.
# Uso overwrite perché questo notebook inizializza da zero lo storico meteo.
df_storico.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(
    tabella_storico
)


# Controllo finale.
# Mostro un campione ordinato della tabella appena scritta.
display(
    spark.table(tabella_storico)
    .orderBy("Data", "Ora", "Citta")
    .limit(100)
)


# Stampo un riepilogo finale del caricamento.
print(f"Bootstrap completato. Tabella aggiornata: {tabella_storico}")
print(f"Righe salvate: {spark.table(tabella_storico).count()}")